# 06 — Optimasi Itinerary: GA vs PSO vs GA-PSO Hybrid

FASE 2 riset (TTDP — Tourist Trip Design Problem): dari kandidat venue hasil
Content-Based Filtering (TF-IDF), tiga algoritma menyusun **itinerary multi-hari**
optimal — dibandingkan sesuai tujuan riset:

| Algoritma | Pendekatan |
|---|---|
| **GA** | tournament selection, Order Crossover (OX), swap mutation, elitism |
| **PSO diskrit** | swap-sequence velocity (adaptasi PSO ke ruang permutasi) |
| **GA-PSO Hybrid** | PSO + refresh genetik tiap N iterasi (crossover pbest×gbest) |

**Encoding**: permutasi kandidat venue → *time-budget decoding* (akumulasi
travel + time_spent per hari, mulai/selesai di hotel, cek jam buka).

**Fitness** = Σ satisfaction − w·travel − w·cross_zone − penalti pelanggaran jam
(satisfaction = w_sim·CBF + w_pop·rating Bayesian).

Kode inti di `05_modeling/` (`problem.py`, `cbf.py`, `ga.py`, `pso.py`,
`hybrid.py`, `experiment.py`) — notebook ini memanggil & memvisualisasikan.

In [ ]:
import os, sys
sys.stdout.reconfigure(encoding='utf-8', errors='replace')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, '05_modeling'))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import config
from problem import TTDPProblem
from cbf import ContentBasedFilter
from ga import run_ga
from pso import run_pso
from hybrid import run_hybrid

print('Root:', ROOT)
print(f'GA pop={config.GA_POP_SIZE} gen={config.GA_N_GEN} | '
      f'PSO particles={config.PSO_N_PARTICLES} iter={config.PSO_N_ITER} | '
      f'Hybrid refresh tiap {config.HYBRID_GA_REFRESH_EVERY}')

---
## 1. Demo: Input Turis → Itinerary

Contoh: turis suka **museum & sejarah**, budget **menengah**, **2 hari**
(Sabtu-Minggu), menginap di hotel dekat pusat kota.

In [ ]:
# Input turis
PREFERENSI  = 'museum sejarah budaya'
BUDGET      = 'menengah'      # hemat / menengah / bebas
N_HARI      = 2
HARI_MULAI  = 'Sabtu'

# Pilih hotel (contoh: hotel dgn rating tertinggi)
hotels = pd.read_csv(config.HOTELS_CSV)
hotel = hotels.sort_values('google_rating', ascending=False).iloc[0]
print(f"Hotel: {hotel['name']} (rating {hotel['google_rating']})")

# FASE 1 — CBF: kandidat top-N + skor satisfaction
cbf = ContentBasedFilter()
ids, sat = cbf.candidates(N_HARI, PREFERENSI, BUDGET)
print(f'Kandidat CBF: {len(ids)} venue')
cbf.score(PREFERENSI, BUDGET).head(10)

In [ ]:
# FASE 2 — Optimasi dengan algoritma terbaik hasil eksperimen (GA tuned)
# verbose=True: progress best-fitness tiap 50 generasi tampil real-time
prob = TTDPProblem(ids, hotel=hotel, n_days=N_HARI, start_day=HARI_MULAI,
                   satisfaction=sat)
res = run_ga(prob, verbose=True)
d = prob.decode(res['best_perm'])

print(f"\nFitness: {res['best_fitness']:.3f} | venue terkunjungi: {len(d['visited'])} "
      f"| travel: {d['travel_total']:.0f} mnt | pelanggaran jam: {d['violations']}")
print()
for di, day in enumerate(d['days']):
    print(f"=== Hari {di+1} ({prob.day_names[di]}) — mulai & pulang: {prob.hotel_name} ===")
    for v in day:
        s, e = int(v['start']), int(v['depart'])
        mark = '  [!] melanggar jam tutup' if v['violation'] else ''
        print(f"  {s//60:02d}:{s%60:02d}-{e//60:02d}:{e%60:02d}  {v['name']}{mark}")
    print()

In [ ]:
# Peta rute per hari (folium)
import folium

m = folium.Map(location=[prob.hotel_lat, prob.hotel_lon], zoom_start=12)
folium.Marker([prob.hotel_lat, prob.hotel_lon], tooltip=f'HOTEL: {prob.hotel_name}',
              icon=folium.Icon(color='red', icon='home')).add_to(m)
colors = ['blue', 'green', 'purple', 'orange', 'darkred']
venues_idx = prob.venues
for di, day in enumerate(d['days']):
    color = colors[di % len(colors)]
    pts = [(prob.hotel_lat, prob.hotel_lon)]
    for order, v in enumerate(day, 1):
        r = venues_idx.loc[v['venue_id']]
        pts.append((r['latitude'], r['longitude']))
        folium.Marker([r['latitude'], r['longitude']],
                      tooltip=f"H{di+1}-{order}: {v['name']}",
                      icon=folium.Icon(color=color)).add_to(m)
    pts.append((prob.hotel_lat, prob.hotel_lon))
    folium.PolyLine(pts, color=color, weight=3, opacity=0.7,
                    tooltip=f'Hari {di+1}').add_to(m)
m

---
## 2. Perbandingan Konvergensi — 1 Run (seed sama)

Kurva best-fitness per generasi/iterasi, ketiga algoritma pada problem yang sama.

In [ ]:
# verbose=True: progress konvergensi tiap 50 generasi/iterasi per algoritma
results_single = {}
for name, fn in [('GA', run_ga), ('PSO', run_pso), ('Hybrid', run_hybrid)]:
    print(f'--- {name} ---')
    results_single[name] = fn(prob, seed=config.RANDOM_SEED, verbose=True)

fig, ax = plt.subplots(figsize=(9, 5))
for name, r in results_single.items():
    ax.plot(r['history'], label=f"{name} (akhir: {r['best_fitness']:.2f})")
ax.set_xlabel('Generasi / Iterasi')
ax.set_ylabel('Best fitness')
ax.set_title(f'Konvergensi GA vs PSO vs Hybrid — {len(ids)} kandidat, {N_HARI} hari')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Eksperimen Penuh — 3 skenario × 3 algoritma × N run

Dijalankan via `05_modeling/experiment.py` (cache-aware: skip kalau hasil ada).
Skenario: `1hari_museum`, `3hari_keluarga`, `5hari_campuran`.

Metrik: fitness (mean±std), **User Satisfaction Score** (USS = rasio satisfaction
terkunjungi vs top-K ideal, 0-1), pelanggaran jam, runtime.

In [ ]:
# [RUN] eksperimen penuh (cache-aware — hapus CSV untuk rerun)
if os.path.exists(config.EXPERIMENT_RESULTS_CSV):
    print(f'[skip] {config.EXPERIMENT_RESULTS_CSV} sudah ada.')
else:
    from experiment import run_experiments
    run_experiments()

In [ ]:
# [STATS] tabel hasil
res = pd.read_csv(config.EXPERIMENT_RESULTS_CSV)
summary = (res.groupby(['scenario', 'algorithm'])
              .agg(fitness_mean=('fitness', 'mean'), fitness_std=('fitness', 'std'),
                   uss=('uss', 'mean'), n_visited=('n_visited', 'mean'),
                   travel_min=('travel_min', 'mean'),
                   violations=('violations', 'mean'),
                   runtime_s=('runtime_sec', 'mean'))
              .round(3))
print(summary.to_string())

In [ ]:
# [STATS] kurva konvergensi rata-rata ± std per skenario
conv = pd.read_csv(config.CONVERGENCE_LOG_CSV)
scenarios = conv['scenario'].unique()
fig, axes = plt.subplots(1, len(scenarios), figsize=(6 * len(scenarios), 4.5),
                         squeeze=False)
for ax, sc in zip(axes[0], scenarios):
    sub = conv[conv['scenario'] == sc]
    for algo in ['GA', 'PSO', 'Hybrid']:
        g = (sub[sub['algorithm'] == algo]
             .groupby('generation')['best_fitness'].agg(['mean', 'std']))
        ax.plot(g.index, g['mean'], label=algo)
        ax.fill_between(g.index, g['mean'] - g['std'], g['mean'] + g['std'], alpha=0.15)
    ax.set_title(sc)
    ax.set_xlabel('Generasi / Iterasi')
    ax.set_ylabel('Best fitness')
    ax.legend()
plt.suptitle('Konvergensi rata-rata ± std', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# [STATS] boxplot fitness akhir + bar USS & runtime
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
order = ['GA', 'PSO', 'Hybrid']

data = [res[res['algorithm'] == a]['fitness'] for a in order]
axes[0].boxplot(data, tick_labels=order)
axes[0].set_title('Distribusi fitness akhir (semua skenario)')
axes[0].set_ylabel('Fitness')

uss_mean = res.groupby('algorithm')['uss'].mean().reindex(order)
axes[1].bar(order, uss_mean, color=['steelblue', 'darkorange', 'seagreen'])
axes[1].set_title('User Satisfaction Score (mean)')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(uss_mean):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center')

rt = res.groupby('algorithm')['runtime_sec'].mean().reindex(order)
axes[2].bar(order, rt, color=['steelblue', 'darkorange', 'seagreen'])
axes[2].set_title('Runtime rata-rata per run (detik)')
for i, v in enumerate(rt):
    axes[2].text(i, v * 1.02, f'{v:.1f}s', ha='center')
plt.tight_layout()
plt.show()

---
## 4. Hyperparameter Tuning — Grid Search

Parameter awal = nilai standar literatur. Grid search (`05_modeling/tune.py`) mencari
parameter terbaik per algoritma dengan **fair comparison**: budget komputasi sama
(pop/particles=50, gen/iter=200), fitness function sama, diuji di 2 skenario ekstrem
(1 hari kecil + 5 hari besar), 5 run per kombinasi, pemilihan via **mean rank**
lintas skenario (robust terhadap beda skala fitness).

| Grid | Kombinasi |
|---|---|
| GA: mutation × crossover | {0.1, 0.2, 0.3} × {0.7, 0.8, 0.9} |
| PSO: w × c1=c2 | {0.4, 0.5, 0.7} × {1.0, 1.5, 2.0} |
| Hybrid: refresh (pakai w/c terbaik PSO) | {5, 10, 20} |

**Hasil tuning (2026-07-04)**: GA `mut=0.3, cx=0.7` | PSO `w=0.4, c=1.0` |
Hybrid `refresh=5`. Sudah diterapkan di `config.py` — eksperimen section 3 di atas
memakai parameter tuned ini.

In [ ]:
# [RUN] grid search tuning (cache-aware — hapus CSV untuk rerun, ~10-15 menit)
TUNING_CSV = 'data/processed/tuning_results.csv'
if os.path.exists(TUNING_CSV):
    print(f'[skip] {TUNING_CSV} sudah ada.')
else:
    import tune
    tune.main()

In [ ]:
# [STATS] hasil grid search — mean rank per kombinasi parameter
tun = pd.read_csv(TUNING_CSV)
tun['rank'] = tun.groupby(['algorithm', 'scenario'])['fitness_mean'].rank(ascending=False)
rank_tbl = (tun.groupby(['algorithm', 'params'])['rank'].mean()
               .sort_values().groupby('algorithm').head(5))
print('Top-5 kombinasi per algoritma (mean rank, kecil = baik):')
print(rank_tbl.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, algo in zip(axes, ['GA', 'PSO', 'Hybrid']):
    sub = tun[tun.algorithm == algo]
    piv = sub.pivot_table(index='params', columns='scenario', values='fitness_mean')
    piv.plot(kind='barh', ax=ax, legend=(algo == 'GA'))
    ax.set_title(f'{algo} — fitness per kombinasi')
    ax.set_xlabel('Fitness mean (5 run)')
plt.tight_layout()
plt.show()

In [ ]:
# [STATS] Before/After tuning — dampak parameter terhadap hasil akhir
base_p = 'data/processed/optimization_results_base.csv'
if os.path.exists(base_p):
    base = pd.read_csv(base_p)
    tuned = pd.read_csv(config.EXPERIMENT_RESULTS_CSV)

    def summar(df, tag):
        s = (df.groupby(['scenario', 'algorithm'])
               .agg(fit=('fitness', 'mean'), viol=('violations', 'mean')).round(3))
        s.columns = [f'{c}_{tag}' for c in s.columns]
        return s

    m = summar(base, 'base').join(summar(tuned, 'tuned'))
    m['delta_fitness'] = (m['fit_tuned'] - m['fit_base']).round(3)
    print(m[['fit_base', 'fit_tuned', 'delta_fitness', 'viol_base', 'viol_tuned']].to_string())

    fig, ax = plt.subplots(figsize=(10, 4.5))
    x = np.arange(len(m))
    ax.bar(x - 0.2, m['fit_base'], 0.4, label='Base', color='lightgray')
    ax.bar(x + 0.2, m['fit_tuned'], 0.4, label='Tuned', color='seagreen')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{s}\n{a}' for s, a in m.index], fontsize=8)
    ax.set_ylabel('Fitness mean')
    ax.set_title('Dampak hyperparameter tuning (8/9 sel membaik)')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('optimization_results_base.csv tidak ada — hasil base tidak tersedia utk perbandingan.')

---
## Kesimpulan (setelah hyperparameter tuning)

- **Tuning berhasil**: 8 dari 9 sel (skenario × algoritma) membaik. PSO paling
  drastis (+9.8 fitness di 5 hari) — masalah PSO base bukan cacat fundamental,
  melainkan inertia `w=0.7` terlalu tinggi untuk representasi swap-sequence;
  `w=0.4` membuat gerakan partikel lebih halus di permutasi panjang.
- **GA (tuned: mut=0.3, cx=0.7) unggul di ketiga skenario** + runtime tercepat —
  paling robust & skalabel untuk TTDP multi-hari.
- **Hybrid** kompetitif di problem menengah-besar (perbaikan +5.3 di 5 hari) tapi
  ada trade-off parameter: `refresh=5` optimal untuk problem besar namun mengurangi
  stabilitas di problem kecil (base `refresh=10` dulu std 0.027 di 1 hari) —
  bahan diskusi *parameter sensitivity*; solusi praktis: refresh adaptif per
  ukuran problem.
- USS ketiga algoritma tinggi (0.85–0.96) — pemilihan venue seragam bagus (CBF
  bekerja); pembeda utama antar algoritma = efisiensi rute & kepatuhan jam buka.
- Hasil lengkap: `optimization_results.csv` (tuned), `optimization_results_base.csv`
  (pembanding), `tuning_results.csv` (grid), `optimization_convergence.csv` — siap
  untuk tabel & grafik paper.